# core

> The error type, the environment convention, and the one place that decides which model runs what.

Every other module depends on this one, and it depends on nothing in the package. Three small things live here because everything needs them. `AgentError`, `agent_err` and `env`. Then the model policy, which routes each job to a model rather than running everything on one.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_ne, test_fail

## Errors and environment

A harness fails in two ways, and a person needs to be told them differently. `AgentError` is a refusal. Something the harness will not do, which the user can act on. Anything else is a bug, and should look like one.

In [ ]:
#| export
import difflib, functools, importlib, importlib.util, json, os, platform, re, shutil, subprocess, sys, threading, time
from fastcore.all import Path
from shalya.host import HostError
from dataclasses import dataclass, field

In [ ]:
#| export
ENV_PREFIX, ENV_FALLBACK = 'RAMABANA_', 'LEELA_'

#: shalya raises `HostError`, and this is that class under the name the harness has always used.
#: An alias rather than a subclass, so `except AgentError` still catches what a host refuses.
AgentError = HostError
class BranchChanged(AgentError): "A branch moved while a person was deciding what to do to it, so nothing was written."

def agent_err(e):
    "A caught exception for a user-facing harness surface."
    return f'{type(e).__name__}: {e}'

def use_env_prefix(prefix, fallback=None):
    "Name the environment variables this application reads, most specific first. `use_env_prefix('LEELA_', 'RAMABANA_')`"
    global ENV_PREFIX, ENV_FALLBACK
    ENV_PREFIX = prefix if prefix.endswith('_') else prefix + '_'
    if fallback is not None: ENV_FALLBACK = fallback if fallback.endswith('_') else fallback + '_'
    return ENV_PREFIX, ENV_FALLBACK

def env(name, dflt=None):
    "`$<prefix><name>`, then `$<fallback><name>`, then `dflt`. See `use_env_prefix`."
    return os.environ.get(ENV_PREFIX+name) or os.environ.get(ENV_FALLBACK+name) or dflt

`agent_err` renders a caught exception for a surface a person reads, keeping the type name because "no such file" without `FileNotFoundError` in front of it explains nothing.

In [ ]:
agent_err(FileNotFoundError('nbs/99_missing.ipynb'))

'FileNotFoundError: nbs/99_missing.ipynb'

`env` reads `$RAMABANA_<name>` first and `$LEELA_<name>` second. A Leela user's existing configuration keeps working while the new prefix takes precedence.

In [ ]:
os.environ['LEELA_MODEL'] = 'opus'
env('MODEL')

'opus'

In [ ]:
os.environ['RAMABANA_MODEL'] = 'sonnet'
test_eq(env('MODEL'), 'sonnet')
del os.environ['RAMABANA_MODEL'], os.environ['LEELA_MODEL']
env('MODEL', 'gemma-e4b')

'gemma-e4b'

## The model tables

`MODELS` maps shared short names to LiteRT models on device or cloud model ids. `JOBS` is the set of distinct jobs the policy can route separately.

In [ ]:
#| export
JOBS = ('turn', 'oneshot', 'inline', 'completion', 'classify', 'summary', 'subagent')
ONESHOT_JOBS = ('oneshot', 'completion', 'classify', 'inline')
LOCAL = {'gemma-e2b': 'litert-community/gemma-4-E2B-it-litert-lm','gemma-e4b': 'litert-community/gemma-4-E4B-it-litert-lm','gemma-12b': 'litert-community/gemma-4-12B-it-litert-lm'}
MLX = {'qwen-4b': 'mlx-community/Qwen3.5-4B-MLX-4bit','mini-coder-4b': 'mlx-community/mini-coder-4b-OptiQ-4bit','ornith-9b': 'mlx-community/Ornith-1.0-9B-8bit'}
LLAMA = {'llama-qwen-0.6b': 'Qwen/Qwen3-0.6B-GGUF','llama-qwen-1.7b': 'Qwen/Qwen3-1.7B-GGUF','llama-qwen-4b': 'Qwen/Qwen3-4B-GGUF'}
GPT = {name: f'openai/{name}' for name in ('gpt-4.1', 'gpt-4.1-mini', 'gpt-4.1-nano','gpt-5.4', 'gpt-5.4-mini', 'gpt-5.6', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra')}
GPT.update({name: f'codex/{name}' for name in ('gpt-5.3-codex-spark', 'gpt-5.5')})
CLOUD = {**GPT, 'gpt': GPT['gpt-5.6-terra'],'gpt-mini': GPT['gpt-5.6-luna'], 'gpt-sol': GPT['gpt-5.6-sol']}
CLAUDE_MODELS = ('claude-opus-5', 'claude-sonnet-5', 'claude-haiku-4-5', 'claude-fable-5','claude-opus-4-8', 'claude-sonnet-4-6', 'claude-sonnet-4-5', 'claude-opus-4-6')
CLAUDE = {f'claude/{mid}': mid for mid in CLAUDE_MODELS}
CLAUDE_ALIASES = {**{mid: mid for mid in CLAUDE_MODELS},'sonnet': 'claude-sonnet-5', 'opus': 'claude-opus-5', 'fable': 'claude-fable-5'}
DFLT_AGENT_CTX = 128_000
CLAUDE_CTX = {'claude-opus': 200_000, 'claude-sonnet': 200_000}
RUNTIMES = ('litert', 'mlx', 'llama', 'ollama', 'claude', 'copilot', 'remote')
AGENTS = ('claude',)
HOSTED = ('remote', 'copilot', *AGENTS)
COPILOT_UNAVAILABLE = ('copilot runtime is unavailable; sign in to Copilot in an editor or run `python -c "from rishi.copilot import copilot_login; copilot_login()"`')
CUSTOM = {}
_RUNTIME_DEPS = {'litert': 'litert_lm', 'mlx': 'mlx_lm', 'llama': 'llama_cpp'}
RUNTIME_REMEDY = {
    'remote': 'hosted models need an API key in the environment',
    'claude': 'install Claude Code (https://claude.com/claude-code) and run `claude /login`',
    'copilot': 'sign in to GitHub Copilot in an editor, or run `copilot_login()` from rishi.copilot',
    'litert': 'LiteRT ships with rishi; reinstall it with `pip install --upgrade rishi`',
}
MODELS = {**{k: ('litert', v) for k, v in LOCAL.items()}, **{k: ('mlx', v) for k, v in MLX.items()},**{k: ('llama', v) for k, v in LLAMA.items()},**{k: ('claude', v) for k, v in {**CLAUDE, **CLAUDE_ALIASES}.items()},**{k: ('remote', v) for k, v in CLOUD.items()}}
PII_OFF = 'off'
PII_MODES = (PII_OFF, 'redact', 'refuse')

In [ ]:
#| export
def claude_ctx(model_id):
    "What a Claude Code model holds, or `DFLT_AGENT_CTX` when its window is not known here."
    mid = str(model_id or '')
    return next((c for p, c in CLAUDE_CTX.items() if mid.startswith(p)), DFLT_AGENT_CTX)

In [ ]:
#| export
#: An agent harness is a module plus the callable that finds the binary its SDK spawns.
#: `runtime_detail` reads the table `_harness_available` answers from, so the yes/no and the reason
#: cannot drift apart.
HARNESS = {'claude': ('rishi.claude', 'claude_bin')}

def _harness_available(mod, binary):
    "Whether an agent harness can be reached: its module imports, and the binary its SDK spawns."
    try: m = importlib.import_module(mod)
    except Exception: return False
    try: return bool(getattr(m, binary)())
    except Exception: return False

def runtime_detail(runtime):
    """Why a harness cannot be reached here, or `''` when it can be or is not a harness.

    `runtime_available` answers yes or no and swallows the reason, which is how a harness that is
    installed but broken reads as "not installed" with nothing naming the cause.
    """
    got = HARNESS.get(runtime)
    if got is None: return ''
    mod, binary = got
    try: m = importlib.import_module(mod)
    except Exception as e: return f'import {mod}: {agent_err(e)}'
    try: found = getattr(m, binary)()
    except Exception as e: return f'{binary}(): {agent_err(e)}'
    return '' if found else f'{binary}() found nothing'

## Probes

Asking what is installed and who is signed in reaches for every backend and every account, which is
slow enough to be most of a picker's first paint. A probe keeps the last answer, serves it at once,
and refreshes a stale one behind whoever asked.

In [ ]:
#| export
PROBE_TTL = 900         #: seconds an answer about this machine stays fresh
PROBE_DIR = 'probes'    #: under the caller's config directory

_probes, _probe_lock, _probing = {}, threading.Lock(), set()
_probe_age = 0          #: bumped by `forget_probes`, so an answer gathered before it is discarded

def probe_path(key, dir=None):
    "Where one probe's last answer is kept between runs."
    d = Path(dir) if dir else Path.home()/'.ramabana'/PROBE_DIR
    return d/f"{re.sub(r'[^A-Za-z0-9_.-]', '_', str(key))}.json"

def _probe_read(key, dir):
    "The answer left on disk, or None."
    try: got = json.loads(probe_path(key, dir).read_text())
    except Exception: return None
    return got if isinstance(got, dict) and 'at' in got else None

def _probe_write(key, value, dir, age):
    "Keep `value`, in memory and on disk, unless the cache was dropped while it was gathered."
    row = {'at': time.time(), 'value': value}
    with _probe_lock:                         # the file is written under the lock `forget_probes`
        if age != _probe_age: return          # empties it under, or a refresh already gathering
        _probes[str(key)] = row               # puts the dropped answer back after the unlink
        p = probe_path(key, dir)
        try:
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_text(json.dumps(row, default=str))
        except Exception: pass

def _probe_behind(key, fn, dir, age):
    "Refresh a stale answer on a daemon thread, so the caller waits for nothing."
    def work():
        try: _probe_write(key, fn(), dir, age)
        except Exception: pass
        finally:
            with _probe_lock: _probing.discard(str(key))
    with _probe_lock:
        if str(key) in _probing: return
        _probing.add(str(key))
    threading.Thread(target=work, daemon=True, name=f'ramabana-probe-{key}').start()

def probed(key, fn, ttl=PROBE_TTL, dir=None):
    "`fn()`'s last answer, with a stale one refreshed behind whoever asked."
    key, warm = str(key), True
    with _probe_lock:
        row, age = _probes.get(key), _probe_age
    if row is None and (row := _probe_read(key, dir)) is not None:
        warm = False        # a previous process learned it: a starting point rather than news
        with _probe_lock:
            if age == _probe_age: _probes[key] = row
    if row is None:
        got = fn()
        _probe_write(key, got, dir, age)
        return got
    if not warm or time.time() - float(row.get('at') or 0) > float(ttl):
        _probe_behind(key, fn, dir, age)
    return row.get('value')

def forget_probes(disk=False, dir=None):
    "Drop every cached probe. What it holds is the machine's, so it is process-wide."
    global _probe_age
    with _probe_lock:
        _probe_age += 1               # an answer already gathering is about the machine before this
        n = len(_probes)
        _probes.clear()
        _probing.clear()
        if disk:
            for p in (Path(dir) if dir else Path.home()/'.ramabana'/PROBE_DIR).glob('*.json'):
                try: p.unlink()
                except Exception: pass
    return n

In [ ]:
#| export
def _claude_available(): return _harness_available(*HARNESS['claude'])

In [ ]:
#| export
def _copilot_available():
    "Checks if Copilot is reachable by verifying the GitHub OAuth token and reading environment/editor config files, without making network calls."
    try:
        from rishi.copilot import copilot_oauth as co
        return bool(co())
    except Exception: return False
        

In [ ]:
#| export
def runtime_remedy(runtime):
    "One sentence saying what to do about a runtime that cannot be reached here."
    return RUNTIME_REMEDY.get(runtime, f'install the backend with `pip install rishi[{runtime}]`')

In [ ]:
#| export
@functools.lru_cache(maxsize=1)
def _ollama_available():
    "Whether ollama can serve here: the daemon is answering, or its binary is on hand."
    try:
        from rishi.ollama import OllamaClient, ollama_bin
        try: return bool(OllamaClient().models() is not None)
        except Exception: return bool(ollama_bin())
    except Exception: return False

In [ ]:
#| export
def runtime_available(runtime):
    "Whether Rishi's optional dependency for `runtime` can be reached. Never raises."
    if runtime == 'remote': return True
    if runtime == 'claude': return _claude_available()
    if runtime == 'copilot': return _copilot_available()
    if runtime == 'ollama': return _ollama_available()
    try: return importlib.util.find_spec(_RUNTIME_DEPS[runtime]) is not None
    except (ImportError, KeyError, ValueError): return False

A name resolves to either the LiteRT runtime or a cloud transport.

In [ ]:
{name: MODELS[name] for name in ('gemma-e4b', 'gpt-4.1-mini', 'claude-sonnet-4-5', 'sonnet')}

{'gemma-e4b': ('litert', 'litert-community/gemma-4-E4B-it-litert-lm'),
 'gpt-4.1-mini': ('remote', 'openai/gpt-4.1-mini'),
 'claude-sonnet-4-5': ('claude', 'claude-sonnet-4-5'),
 'sonnet': ('claude', 'claude-sonnet-5')}

## Credentials

`auth_status` reports available credential sources from status metadata. It never returns, logs, or stores a secret. The Claude Code probe calls the `claude` binary instead of reading its credential file.

In [ ]:
#| export
def _json_has(path, *keys):
    "Whether nested keys in a JSON file are present and truthy."
    try:
        from fastcore.basics import nested_idx
        return bool(nested_idx(Path(path).expanduser().read_json(), *keys))
    except Exception: return False

def _claude_login():
    "Only status metadata. Credentials never leave Claude Code or enter Leela."
    if not shutil.which('claude'): return False
    try:
        p = subprocess.run(['claude', 'auth', 'status', '--json'], capture_output=True, text=True, timeout=3)
        return p.returncode == 0 and bool(json.loads(p.stdout).get('loggedIn'))
    except Exception: return False

def auth_status():
    'Credential sources FastLLM can use, without reading or returning any secret.'
    codex = bool(os.getenv('CODEX_AUTH_TOKEN') or _json_has(os.getenv('CODEX_AUTH_PATH', '~/.codex/auth.json'), 'tokens', 'access_token'))
    claude_login = _claude_login() and _claude_available()
    copilot = _copilot_available()
    out = {v: {'available': bool(os.getenv(k)), 'source': k} for v, k in API_KEYS.items()}
    return out | {
        'codex': {'available': codex, 'source': 'Codex login' if codex else ''},
        'claude': {'available': claude_login, 'source': 'Claude Code login' if claude_login else ''},
        'copilot': {'available': copilot, 'source': 'GitHub Copilot sign-in' if copilot else ''},
    }

The shape is the same for every vendor whether or not it is connected. A caller can render the whole list without special cases.

In [ ]:
{k: v['available'] for k, v in auth_status().items()}

{'openai': True,
 'codex': True,
 'anthropic': False,
 'claude': True,
 'gemini': True,
 'copilot': True}

## Listing models

`available_models` is what a model picker renders. It lists the on-device models unconditionally, adds llama.cpp only when that package is importable, and adds a vendor's cloud catalog only when a credential for it exists. The list never offers a model the process cannot actually run.

In [ ]:
#| export
_oai_cache = (0.0, [])
def _openai_models(include_legacy=False):
    "Canonical models the current OpenAI key can list. Older coding models are opt-in."
    global _oai_cache
    ids = _oai_cache[1] if (time.time() - _oai_cache[0]) < 300 else None
    if not (key := os.getenv('OPENAI_API_KEY')): return []
    if ids is None:
        try:
            import httpx2 as httpx
            r = httpx.get('https://api.openai.com/v1/models', headers={'Authorization': f'Bearer {key}'}, timeout=10)
            r.raise_for_status()
            ids = [x.get('id', '') for x in r.json().get('data', [])]
        except Exception: ids = []
        _oai_cache = (time.time(), ids)
    current = re.compile(r'^(?:gpt-5(?:\.\d+)?(?:-(?:mini|nano|pro|codex(?:-mini|-max)?|chat-latest|search-api|[a-z]+))?|o[34](?:-mini|-pro)?)$')
    legacy = re.compile(r'^gpt-4\.1(?:-mini|-nano)?$')
    return sorted({x for x in ids if (current.match(x) or (include_legacy and legacy.match(x))) and not re.search(r'-20\d\d-', x)})

_copilot_cat = (0., {})
def copilot_catalog(ttl=300):
    "Copilot's catalogue for this account, cached: `{id: entry}`, or `{}` when it cannot be reached."
    global _copilot_cat
    if (time.time() - _copilot_cat[0]) < ttl: return _copilot_cat[1]
    try:
        from rishi.copilot import copilot_catalog as cat
        d = cat()
    except Exception: d = {}
    _copilot_cat = (time.time(), d)
    return d

def _copilot_chat_models():
    "Chat ids this Copilot plan can reach. Per-plan and it moves. It is asked for, never tabled."
    return [i for i, m in copilot_catalog().items() if (m.get('capabilities') or {}).get('type') == 'chat']

def available_models(include_legacy=False):
    "Models selectable here. Specialized older generations appear only when requested."
    rows = []
    for runtime, models in (('litert', LOCAL), ('mlx', MLX), ('llama', LLAMA)):
        if not runtime_available(runtime): continue
        rows += [{'value': name, 'label': name, 'provider': runtime, 'source': f'on device via Rishi {runtime}'} for name in models]
    if runtime_available('ollama'):
        try:
            from rishi.ollama import OllamaClient
            for m in OllamaClient().models():          # names, not rows
                rows.append({'value': f'ollama/{m}', 'label': m, 'provider': 'ollama', 'source': 'on device via the ollama daemon'})
        except Exception: pass
    if runtime_available('claude'):
        rows += [{'value': name, 'label': mid, 'provider': 'claude', 'source': 'Claude Code (CLI or SDK)'} for name, mid in CLAUDE.items()]
    if runtime_available('copilot'):
        for model in _copilot_chat_models():
            rows.append({'value': f'copilot/{model}', 'label': model, 'provider': 'copilot', 'source': 'GitHub Copilot subscription'})
    auth = auth_status()
    if auth['openai']['available']:
        for model in _openai_models(include_legacy):
            rows.append({'value': f'openai/{model}', 'label': model, 'provider': 'openai', 'source': auth['openai']['source']})
    try:
        from fastllm.types import model_info_registry
        vendors = ('openai', 'codex', 'gemini')
        for vendor in vendors:
            if not auth.get(vendor, {}).get('available'): continue
            for v, model in model_info_registry:
                if v != vendor: continue
                if not include_legacy and re.match(r'^gpt-4(?:\.|-|$)', model): continue
                rows.append({'value': f'{vendor}/{model}', 'label': model, 'provider': vendor, 'source': auth[vendor]['source']})
    except Exception: pass
    if auth['anthropic']['available']:
        catalog = set()
        try:
            from fastllm.types import model_info_registry
            catalog = {model for vendor, model in model_info_registry if vendor == 'anthropic' and model.startswith('claude-')}
        except Exception: pass
        catalog.update(model.split('/', 1)[1] for model in CLOUD.values() if model.startswith('anthropic/'))
        for model in sorted(catalog): rows.append({'value': f'anthropic/{model}', 'label': model, 'provider': 'anthropic', 'source': auth['anthropic']['source']})
    seen, out = set(), []
    for row in rows:
        if row['value'] in seen: continue
        seen.add(row['value']); out.append(row)
    return out

Every row carries the `source` that made it available, which is how the picker explains itself. The local rows are always present:

In [ ]:
_orig_avail, _orig_cat = _copilot_available, copilot_catalog
_copilot_available = lambda: True
copilot_catalog = lambda ttl=300: {'test-chat': {'capabilities': {'type': 'chat'}}}
try:
    rows = available_models()
    test_eq([r['value'] for r in rows if r['provider'] == 'litert'], list(LOCAL))
    test_eq([r['value'] for r in rows if r['provider'] == 'copilot'], ['copilot/test-chat'])
finally: _copilot_available, copilot_catalog = _orig_avail, _orig_cat

Older coding models are opt-in rather than absent, since `include_legacy=True` is the only way to get a `gpt-4.1` back.

In [ ]:
names = {r['value'] for r in available_models()}
legacy = {r['value'] for r in available_models(include_legacy=True)}
test_eq(names - legacy, set())
sorted(legacy - names)

['openai/gpt-4.1', 'openai/gpt-4.1-mini', 'openai/gpt-4.1-nano']

## Defaults and context windows

LiteRT is the only local runtime. `DEFAULT_POLICY` leaves `turn` unset. That is the user's choice. Points the cheap, frequent jobs at the small local Gemma.

In [ ]:
#| export
DFLT_LOCAL = 'gemma-e4b'
completer = DFLT_LOCAL
cheap = completer          # back-compat alias

DEFAULT_POLICY = {'turn': None, 'oneshot': completer, 'inline': None, 'completion': None, 'classify': None, 'summary': None, 'subagent': 'gpt-4.1'}
_LOCAL_CTX = {'gemma-e2b': 16_384, 'gemma-e4b': 16_384, 'gemma-12b': 32_000, 'qwen-4b': 32_768, 'mini-coder-4b': 32_768, 'ornith-9b': 32_768, 'llama-qwen-0.6b': 32_768, 'llama-qwen-1.7b': 32_768, 'llama-qwen-4b': 32_768}
DFLT_LOCAL_CTX = 32_768


@functools.lru_cache(maxsize=64)
def local_window(runtime, model_id):
    "What a local model was trained for, from its own config. 0 when it will not say."
    try:
        if runtime == 'ollama':
            from rishi.ollama import OllamaClient
            info = OllamaClient().show(model_id).get('model_info') or {}
            return int(next((v for k, v in info.items() if k.endswith('.context_length')), 0) or 0)
        if runtime == 'mlx':
            from rishi.mlx import ctx_len, read_config
            return int(ctx_len(read_config(model_id), 0) or 0)
        if runtime == 'llama':
            from llama_cpp import Llama
            from rishi.llama import _get_model
            meta = Llama(_get_model(model_id), vocab_only=True, verbose=False).metadata
            return int(meta.get(f"{meta.get('general.architecture')}.context_length") or 0)
    except Exception: pass
    return 0

def local_ctx(name, dflt=DFLT_LOCAL_CTX):
    "How much of a local model to use: `$LEELA_LOCAL_CTX` if it says, else the table."
    if (ovr := (env('LOCAL_CTX') or '').strip()):
        if ovr.isdigit(): return int(ovr)
        for part in ovr.split(','):
            k, _, v = part.partition(':')
            if k.strip() == name and v.strip().isdigit(): return int(v)
    return _LOCAL_CTX.get(name, dflt)

Every job in `JOBS` has an entry. A new job cannot be added without deciding where it runs.

In [ ]:
test_eq(set(JOBS) - set(DEFAULT_POLICY), set())

A local model's context window comes from the table, and `$RAMABANA_LOCAL_CTX` overrides it. Either as one number for everything, or as `name:size` pairs.

In [ ]:
local_ctx('gemma-e4b'), local_ctx('gemma-12b'), local_ctx('a-model-nobody-tabled')

(16384, 32000, 32768)

In [ ]:
os.environ['RAMABANA_LOCAL_CTX'] = 'gemma-e4b:4096'
test_eq(local_ctx('gemma-e4b'), 4096)
test_eq(local_ctx('gemma-12b'), 32_000)          # untouched by a per-model override
del os.environ['RAMABANA_LOCAL_CTX']
local_ctx('gemma-e4b')

16384

## Resolving a model

`ModelSpec` resolves a model name into its runtime, runtime options, context window, and user-facing note. Downstream code accepts a `ModelSpec` rather than resolving the name again.

In [ ]:
#| export
@dataclass(frozen=True)
class ModelSpec:
    'One model, resolved: which backend runs it, what to call it, and how big it is.'
    name: str                 # what the user types
    backend: str              # 'rishi' | 'fastllm'
    model_id: str             # what the backend is given
    ctx: int = 128_000        # context window in tokens
    note: str = ''            # anything worth showing about how this was resolved
    config: dict = field(default_factory=dict, compare=False) # runtime options. Never persisted secrets

    @property
    def runtime(self): return self.backend
    @property
    def local(self): return self.backend not in HOSTED
    def __str__(self): return f'{self.name} ({self.model_id})'

def _copilot_ctx(model_id):
    "Context window and a note for a Copilot model."
    lim = ((copilot_catalog().get(model_id) or {}).get('capabilities') or {}).get('limits') or {}
    if (n := lim.get('max_prompt_tokens') or lim.get('max_context_window_tokens')): return int(n), ''
    return _cloud_ctx(model_id)

def _cloud_ctx(model_id):
    "Context window and a note for a cloud model, from fastllm's tables. Silent about failure."
    try:
        from fastllm.types import get_model_info
        v, _, m = model_id.partition('/')
        info = get_model_info(m or v, v if m else None)
        n = info.get('max_input_tokens') or info.get('max_tokens')
        if n: return int(n), ''
        return 128_000, 'context window unknown, assuming 128k'
    except Exception as e: return 128_000, f'context window unknown ({agent_err(e)}), assuming 128k'

In [ ]:
#| export
def unknown_model(name):
    near = difflib.get_close_matches(str(name), MODELS, n=2, cutoff=0.6)
    hint = f'; did you mean {" or ".join(repr(n) for n in near)}?' if near else '.'
    return (f'unknown model {name!r}{hint} `/models` lists what is configured, and any vendor/model spec works too.')

PREFIXES = RUNTIMES
RETIRED = {'claude_code': 'use `claude/` instead: the same models, through Claude Code itself', 'cursor': 'the Cursor backend was removed'}

def resolve(name, default_local=DFLT_LOCAL):
    'A `ModelSpec` for `name`: a short name from the tables, or any full `vendor/model` spec.'
    if not name: name = default_local
    if name in MODELS:
        backend, mid = MODELS[name]
        config = CUSTOM.get(name, {}).get('config', {})
        if backend not in ('remote', 'copilot'):
            if not runtime_available(backend): raise RuntimeError(f'{backend} runtime is unavailable; {runtime_remedy(backend)}')
            ctx = claude_ctx(mid) if backend == 'claude' else DFLT_AGENT_CTX if backend in AGENTS else local_ctx(name)
            return ModelSpec(name, backend, mid, ctx, config=config)
        if backend == 'copilot':
            ctx, note = _copilot_ctx(mid)
            return ModelSpec(name, backend, mid, ctx, note, config)
        ctx, note = _cloud_ctx(mid)
        return ModelSpec(name, backend, mid, ctx, note, config)
    if '/' in name:
        runtime, model_id = name.split('/', 1)
        if runtime in RETIRED: raise KeyError(f'{name!r}: the {runtime!r} prefix was removed -- {RETIRED[runtime]}')
        if runtime == 'copilot':
            if not runtime_available('copilot'): raise RuntimeError(COPILOT_UNAVAILABLE)
            ctx, note = _copilot_ctx(model_id)
            return ModelSpec(name, 'copilot', model_id, ctx, note)
        if runtime in ('litert', 'mlx', 'llama', 'ollama', *AGENTS):
            if not runtime_available(runtime): raise RuntimeError(f'{runtime} runtime is unavailable; {runtime_remedy(runtime)}')
            ctx = claude_ctx(model_id) if runtime == 'claude' else DFLT_AGENT_CTX if runtime in AGENTS else local_ctx(name)
            return ModelSpec(name, runtime, model_id, ctx)
        from urai import infer_runtime
        if (inferred := infer_runtime(name)) in ('litert', 'mlx', 'llama'):
            if not runtime_available(inferred): raise RuntimeError(f'{inferred} runtime is unavailable; {runtime_remedy(inferred)}')
            return ModelSpec(name, inferred, name, local_ctx(name))
        ctx, note = _cloud_ctx(name)
        return ModelSpec(name, 'remote', name, ctx, note)
    raise KeyError(unknown_model(name))

@functools.lru_cache(maxsize=256)
def _caps(model_id, runtime):
    "`urai.model_caps`, memoised."
    try:
        from urai import model_caps
        return model_caps(model_id, runtime=runtime)
    except Exception: return None

def spec_caps(spec):
    "What `spec`'s model accepts and what it hands back, or `None` where rishi cannot say."
    return _caps(spec.model_id, spec.backend if spec.local else 'remote')

def accepts(spec, kind):
    "Can `spec`'s model be sent `kind`? Unknown counts as yes."
    c = spec_caps(spec)
    return True if c is None or not c.known else c.accepts(kind)

def model_note(spec):
    "One line about a resolved model, for a status bar."
    where = 'local' if spec.local else 'cloud'
    out = f'{spec.name} · {where} · {spec.ctx//1000}k ctx'
    if (c := spec_caps(spec)) is not None and (m := c.fmt()): out += f' · {m}'
    return out + (f' · {spec.note}' if spec.note else '')

A short name resolves from the tables, and a full `vendor/model` spec resolves even though no table mentions it. A model released this morning is usable this morning.

In [ ]:
resolve('gemma-e4b')

ModelSpec(name='gemma-e4b', backend='litert', model_id='litert-community/gemma-4-E4B-it-litert-lm', ctx=16384, note='', config={})

In [ ]:
spec = resolve('claude/claude-sonnet-5')
test_eq(spec.backend, 'claude')
test_eq(spec.local, False)
model_note(spec)

# an untabled name is inferred from its vendor prefix, where that runtime can actually run. On a
# machine without it, `resolve` refuses with the remedy rather than handing back a dead spec
if runtime_available('mlx'):
    spec = resolve('mlx-community/a-model-nobody-tabled')
    test_eq((spec.backend, spec.model_id), ('mlx', 'mlx-community/a-model-nobody-tabled'))
else:
    test_fail(lambda: resolve('mlx-community/a-model-nobody-tabled'), contains='mlx runtime is unavailable')
test_eq(resolve('somevendor/some-model').backend, 'remote')
test_fail(lambda: resolve('gpt-image-2'), contains='unknown model')

Anything else is a typo, and fails at the point of the typo rather than in the middle of a turn.

In [ ]:
test_fail(lambda: resolve('gpt-9'), contains='unknown model')

Copilot is a runtime of its own, and has to be. Handed to `remote`, `copilot/gpt-5.5` loses its prefix. Rishi reads it as a runtime it already knows and drops it. The bare id goes to whichever vendor owns that name, on that vendor's key. The turn succeeds and the bill is a surprise. The prefix resolves here, and the catalogue is asked for rather than tabled: it is per-plan, it moves, and each entry carries the window Copilot will actually allow.

In [ ]:
# nothing below reaches GitHub: the sign-in and the catalogue are both answered from here
_cat = {'gpt-5.5': {'capabilities': {'type': 'chat', 'limits': {'max_prompt_tokens': 272000}}},
        'claude-opus-4.7': {'capabilities': {'type': 'chat', 'limits': {'max_context_window_tokens': 264000}}},
        'text-embedding-3-small': {'capabilities': {'type': 'embeddings'}}}
_orig_avail, _orig_cat = _copilot_available, copilot_catalog
_copilot_available, copilot_catalog = (lambda: True), (lambda ttl=300: _cat)
try:
    test_eq(runtime_available('copilot'), True)
    test_eq(auth_status()['copilot']['available'], True)

    spec = resolve('copilot/gpt-5.5')
    test_eq(spec.runtime, 'copilot')          # its own runtime, not `remote`
    test_eq(spec.model_id, 'gpt-5.5')         # the prefix named the runtime. It is not part of the id
    test_eq(spec.local, False)                # hosted. It is not sized like something on this disk
    test_eq(spec.ctx, 272000)                 # Copilot's own number
    test_eq(resolve('copilot/claude-opus-4.7').ctx, 264000)   # and its other spelling of one

    # a model Copilot did not name still resolves, on the vendor table, rather than failing here
    assert resolve('copilot/gpt-6-unreleased').ctx > 0

    rows = [r for r in available_models() if r['provider'] == 'copilot']
    test_eq({r['value'] for r in rows}, {'copilot/gpt-5.5', 'copilot/claude-opus-4.7'})
    assert all(r['label'] and r['source'] for r in rows)      # an embedding model cannot answer a turn

finally: _copilot_available, copilot_catalog = _orig_avail, _orig_cat

# and with no sign-in, the ask fails where the typo is rather than three layers downstream
_copilot_available = lambda: False
try: test_fail(lambda: resolve('copilot/gpt-5.5'), contains='sign in to Copilot')
finally: _copilot_available = _orig_avail

## What a model can afford

A briefing is not free, and on a small window it is most of the window. The tool schemas come to 4.7k tokens on a full host, one inlined skill body to 3k more, and the briefing itself to
1.4k. Against the 12.3k a 16k model has before compaction fires. That left room for a single
tool result. The model that ships as the local default could not finish a turn that searched, read, edited and checked: it ran out of window, and `Compactor` had nothing old enough to compact.

`Budget` is that arithmetic, decided from the resolved spec here beside the routing table rather than left to whatever the default constants happen to be. It only ever *withholds*, and nothing it declines is unreachable. A skill body it will not inline is still one `read_skill` away.

In [ ]:
#| export
SMALL_CTX = 24_000       # at or below this window, a model is briefed frugally
TOOL_MAX_FLOOR = 1500    # chars. Below this a file view stops being a file view
FRUGAL_DROP = ('memory', 'web')
TAGS_SCHEMA_TOKENS = 3300

@dataclass(frozen=True)
class Budget:
    'What a model can afford to be told, and to be sent back.'
    drop: tuple = ()         # capability groups to withhold from `tools_for`
    inline: bool = True      # whether the briefing may inline a skill body
    tool_max: int = 0        # chars one tool result may spend
    note: str = ''           # why, for a status bar

def budget_for(spec, tool_max, channel='native'):
    "Restricts tool context to the model's window as defined in `spec`, or to `tool_max` if unset. If the window size can't be determined, defaults to the full briefing. Never increases the context size."
    ctx = getattr(spec, 'ctx', 0) or 0
    if ctx > 0 and channel == 'tags': ctx = max(1, ctx - TAGS_SCHEMA_TOKENS)
    if ctx <= 0 or ctx > SMALL_CTX: return Budget(tool_max=tool_max, note='full briefing')
    mx = min(tool_max, max(TOOL_MAX_FLOOR, (ctx//16)*4))
    return Budget(FRUGAL_DROP, False, mx, f'{ctx//1000}k window: no inlined skills, no {"/".join(FRUGAL_DROP)} tools, tool results clipped to {mx} chars')

In [ ]:
spec = ModelSpec('gemma-e2b', 'litert', 'litert-community/x', 16_384)
b = budget_for(spec, 6000)
test_eq((b.drop, b.inline, b.tool_max), (('memory', 'web'), False, 4096))
b.note

'16k window: no inlined skills, no memory/web tools, tool results clipped to 4096 chars'

In [ ]:
# A 32k local model is briefed in full, and its clip is left where it was.
test_eq(budget_for(ModelSpec('qwen-4b', 'llama', 'x', 32_768), 6000), Budget(tool_max=6000, note='full briefing'))
# An unknown window is not treated as a small one.
test_eq(budget_for(ModelSpec('mystery', 'remote', 'x/y', 0), 6000).inline, True)
# The floor holds for a window small enough that a sixteenth is not a usable view.
test_eq(budget_for(ModelSpec('tiny', 'litert', 'x', 4096), 6000).tool_max, TOOL_MAX_FLOOR)

## Configurable models

`register_model` adds an alias for this process only. Persisting it is the host application's business, since Ramabana does not own the user's configuration file. Passing a Hugging Face URL is enough. Rishi decides which runtime can serve it.

In [ ]:
#| export
def register_model(name, model_id, runtime=None, ctx=128_000, note='custom model', **config):
    "Register a configurable model alias for this process. Persistence belongs to the host app."
    name, model_id = (name or '').strip(), (model_id or '').strip()
    if model_id.startswith(('https://huggingface.co/', 'http://huggingface.co/')): 
        model_id = model_id.split('huggingface.co/', 1)[1].strip('/').split('/tree/', 1)[0]
    if not name: name = model_id.rsplit('/', 1)[-1]
    if not name or not model_id: raise ValueError('model name and model id are required')
    if runtime is None:
        from urai import resolve_runtime
        runtime, model_id = resolve_runtime(model_id)
    if runtime not in RUNTIMES: raise ValueError(f'unknown runtime {runtime!r}')
    if runtime != 'remote' and not runtime_available(runtime): raise RuntimeError(f'{runtime} runtime is unavailable; {runtime_remedy(runtime)}')
    MODELS[name] = (runtime, model_id); _LOCAL_CTX[name] = int(ctx or 128_000)
    CUSTOM[name] = {'name': name, 'model_id': model_id, 'runtime': runtime, 'ctx': int(ctx or 128_000), 'note': note, 'config': config}
    return resolve(name)

def unregister_model(name):
    "Remove one process-local configurable model alias."
    CUSTOM.pop(name, None); MODELS.pop(name, None); _LOCAL_CTX.pop(name, None)

## Saved aliases

`register_model` lasts as long as the process. A frontend that lets somebody name a model wants it
to survive a restart, and both frontends wanted the same file for the same reason. The rows are
here; the path is the caller's, because where an application keeps its config is its own business.

In [ ]:
#| export
#: vendor -> the environment variable holding its key. `auth_status` reads it, and it is the one
#: place the three names are written down.
API_KEYS = {'openai': 'OPENAI_API_KEY', 'anthropic': 'ANTHROPIC_API_KEY', 'gemini': 'GEMINI_API_KEY'}

#: what an alias row keeps. A key itself is never among them: `api_key_env` names the variable to
#: read at use, so a file somebody syncs between machines carries no secret.
MODEL_FIELDS = ('name', 'model_id', 'runtime', 'ctx', 'note')

def _alias_row(spec_row):
    "One saved row: the fields worth keeping, and the configuration that is not a secret."
    cfg = {k: v for k, v in (spec_row.get('config') or {}).items()
           if not any(s in k.lower() for s in ('key', 'token', 'secret', 'password'))
           or k.lower().endswith('_env')}
    return {k: spec_row.get(k) for k in MODEL_FIELDS} | cfg

def _alias_read(path):
    "The rows on disk, or `[]` where there are none."
    try: got = json.loads(Path(path).read_text())
    except Exception: return []
    return [r for r in got if isinstance(r, dict) and r.get('name')] if isinstance(got, list) else []

def _alias_write(rows, path):
    "Replace the file, through a temporary one so a crash mid-write leaves the old rows."
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_suffix(p.suffix + '.tmp')
    tmp.write_text(json.dumps(rows, indent=2, default=str))
    tmp.replace(p)
    return rows

def load_models(path):
    "Register every saved alias in this process, and return the rows that took."
    out = []
    for r in _alias_read(path):
        try:
            register_model(r['name'], r.get('model_id'), r.get('runtime'), r.get('ctx'),
                           r.get('note') or 'custom model',
                           **{k: v for k, v in r.items() if k not in MODEL_FIELDS})
            out.append(r)
        except Exception: pass   # a runtime that is gone is not a reason to lose the rest
    return out

def save_model(row, path):
    "Validate, register and persist one alias. Returns the row as it was stored."
    spec = register_model(row.get('name'), row.get('model_id'), row.get('runtime'), row.get('ctx'),
                          row.get('note') or 'custom model',
                          **{k: v for k, v in row.items() if k not in MODEL_FIELDS})
    kept = _alias_row(CUSTOM[spec.name])
    _alias_write([r for r in _alias_read(path) if r.get('name') != spec.name] + [kept], path)
    return kept

def delete_model(name, path):
    "Forget one alias, on disk and in this process. Returns whether there was one."
    rows = _alias_read(path)
    left = [r for r in rows if r.get('name') != name]
    _alias_write(left, path)
    try: unregister_model(name)
    except Exception: pass
    return len(left) != len(rows)

In [ ]:
register_model('tiny', LOCAL['gemma-e2b'], runtime='litert', ctx=8192)

ModelSpec(name='tiny', backend='litert', model_id='litert-community/gemma-4-E2B-it-litert-lm', ctx=8192, note='', config={})

The alias behaves like a table entry from then on, and `unregister_model` takes it back out of every table it was added to.

In [ ]:
test_eq(resolve('tiny').model_id, LOCAL['gemma-e2b'])
unregister_model('tiny')
test_fail(lambda: resolve('tiny'), contains='unknown model')
'tiny' in MODELS, 'tiny' in CUSTOM

(False, False)

A runtime no engine implements is refused, rather than accepted and discovered later by a turn that cannot start.

In [ ]:
test_fail(lambda: register_model('x', 'some/model', runtime='pytorch'), contains='unknown runtime')

In [ ]:
# a Copilot model can be registered under a short name, which `unknown runtime` used to refuse
_cat = {'gpt-5.5': {'capabilities': {'type': 'chat', 'limits': {'max_prompt_tokens': 272000}}}}
_orig_avail, _orig_cat = _copilot_available, copilot_catalog
_copilot_available, copilot_catalog = (lambda: True), (lambda ttl=300: _cat)
try:
    register_model('cop-test', 'gpt-5.5', runtime='copilot')
    test_eq(resolve('cop-test').runtime, 'copilot')
    test_eq(resolve('cop-test').ctx, 272000)      # sized from the catalogue, not as something local
    test_eq(resolve('cop-test').local, False)
finally:
    _copilot_available, copilot_catalog = _orig_avail, _orig_cat
    MODELS.pop('cop-test', None); CUSTOM.pop('cop-test', None); _LOCAL_CTX.pop('cop-test', None)

## Where the tool schemas travel

A transport's own tool field is the better channel wherever it is open: the schemas are validated, the calls come back structured, and nothing rests on the model minding its punctuation. It is not always open. Claude Code declares tools as an in-process MCP server, and an enterprise-managed configuration forbids every dynamic MCP server there is, which turns a policy about MCP into a coding agent that cannot read a file.

Rishi's `tool_mode='tags'` renders the schemas into the system prompt instead and reads the calls back out of the reply text. That channel cannot be closed. There is no policy against a system prompt. `tool_channel` is the one place that decides, and a Claude Code model becomes a conversational backend like any other rather than one that works only where MCP does.

The agent harnesses sit on both sides of that. Rishi carries Claude Code's tools natively where its SDK can - an in-process MCP server - and on tags where it cannot, which is the CLI and any machine whose policy refuses the server. Only the chat knows which of those it ended up on. `tool_channel` asks it whenever there is one and predicts from the spec only when there is not, which is what `budget_for` has to work from: it sizes the tool list, and the tool list is what builds the chat.

It is also the channel every backend already shares: `parse_tool_tags` predates this, because Hermes-style tag calls are how the local engines have always called tools. The same briefing is portable across litert, llama, MLX and hosted models, which is what makes comparing them on one task an experiment with one variable.

In [ ]:
#| export
TOOL_CHANNELS = ('native', 'tags')
_forced_tags = {}   # model_id -> why its wire tool channel is closed on this machine

def force_tags(model_id, why=''):
    "Record that this model's tools cannot travel on the wire here. Later turns stop trying."
    why = why or 'the wire tool channel was refused'
    _forced_tags[str(model_id)] = why
    return why

def forget_forced_tags():
    "Forget what was learned about wire channels. A fixed configuration is tried again."
    _forced_tags.clear()

def tool_channel(spec, chat=None):
    "Returns the channel ('native' or 'tags') for a model's tool schemas, given a `ModelSpec` or model id and optional live chat."
    if (v := (env('TOOL_CHANNEL') or '').strip().lower()) in TOOL_CHANNELS: return v
    if (ch := getattr(chat, 'tool_channel', None)) in TOOL_CHANNELS: return ch
    mid = str(getattr(spec, 'model_id', spec) or '')
    rt = getattr(spec, 'runtime', '') or (mid.split('/', 1)[0] if '/' in mid else '')
    if rt in AGENTS: return 'tags'
    if mid in _forced_tags: return 'tags'
    return 'native'

In [ ]:
os.environ.pop('RAMABANA_TOOL_CHANNEL', None)      # stated, not inherited: that is the cell's point
test_eq(tool_channel(ModelSpec('sonnet', 'remote', 'claude-sonnet-4-5', 200_000)), 'native')
os.environ['RAMABANA_TOOL_CHANNEL'] = 'tags'      # any model, for a machine we cannot detect
test_eq(tool_channel('gpt-5.1'), 'tags')
del os.environ['RAMABANA_TOOL_CHANNEL']
force_tags('gpt-5.6-terra', 'the wire refused the tool schemas')
test_eq(tool_channel('gpt-5.6-terra'), 'tags')
test_eq(tool_channel('gpt-5.1'), 'native')        # remembered per model, not globally
forget_forced_tags()

class _Chat:
    def __init__(self, ch): self.tool_channel = ch
test_eq(tool_channel(ModelSpec('c', 'claude', 'claude-opus-5', 200_000), _Chat('native')), 'native')
test_eq(tool_channel(ModelSpec('c', 'claude', 'claude-opus-5', 200_000), _Chat('tags')), 'tags')

# Claude is never native: its one channel for a tool it did not ship with is an MCP server, and a
# managed config refuses even the in-process kind. Rishi exposing `tool_channel` does not change that.
import rishi.claude
assert getattr(rishi.claude.ClaudeChat, 'tool_channel', None) is not None   # the attribute is not the claim
test_eq(tool_channel(ModelSpec('sonnet', 'claude', 'claude-sonnet-5', 200_000)), 'tags')
test_eq(tool_channel('claude/claude-sonnet-5'), 'tags')     # and a bare id answers the same
# Copilot is not a harness at all: it is chat completions. The schemas go on the wire
test_eq(tool_channel(ModelSpec('cop', 'copilot', 'gpt-5.5', 272_000)), 'native')

## Routing

`Routing` is job to model, and the only thing that decides what runs where. `turn` is the model the user chose. Every other job falls back to it when the policy has nothing to say. Adding a job to `JOBS` cannot strand a caller without a model.

In [ ]:
#| export
@dataclass
class Routing:
    """Job -> model. The policy, and the one place that decides what runs where.
    Environment overrides (`LEELA_MODEL`, `LEELA_MODEL_SUMMARY`, ...) exist.
    """
    turn: str = None
    policy: dict = field(default_factory=lambda: dict(DEFAULT_POLICY))
    default_local: str = DFLT_LOCAL

    def __post_init__(self):
        if not self.turn: self.turn = env('MODEL') or self.default_local
        for job in JOBS:
            if (v := env(f'MODEL_{job.upper()}')): self.policy[job] = v
        self._cache, self.notes = {}, {}

    def name_for(self, job='turn'):
        """The model name `job` runs on: its own policy, then `oneshot` if it is a cheap job, then `turn`."""
        if job == 'turn': return self.turn
        if (n := self.policy.get(job)): return n
        if job in ONESHOT_JOBS and (n := self.policy.get('oneshot')): return n
        return self.turn

    def _resolve(self, name):
        "One resolution, cached: reading fastllm's tables for a cloud model is not free."
        if name not in self._cache: self._cache[name] = resolve(name, self.default_local)
        return self._cache[name]

    def alternatives(self, job):
        "Where `job` goes when its own model is not on this machine, best first."
        seen, out = {self.name_for(job)}, []
        for alt in (self.policy.get('oneshot') if job in ONESHOT_JOBS else None,
                    self.turn, *(self.policy.get(j) for j in JOBS), self.default_local):
            if alt and alt not in seen:
                seen.add(alt); out.append(alt)
        return out

    def spec(self, job='turn', fallback=True):
        """The resolved `ModelSpec` for `job`, on another model when its own is not installed here."""
        n = self.name_for(job)
        try: return self._resolve(n)
        except Exception as e:
            if not fallback or job == 'turn': raise
            for alt in self.alternatives(job):
                try: spec = self._resolve(alt)
                except Exception: continue
                self.notes[job] = f'{n} unavailable ({agent_err(e)}); using {alt}'
                return spec
            raise

    def set(self, name, job='turn'):
        "Point `job` at `name`, validating it first so a typo fails here rather than mid-turn."
        spec = resolve(name, self.default_local)
        if job == 'turn': self.turn = name
        else: self.policy[job] = name
        self._cache[name] = spec
        return spec

    def backends(self):
        "The distinct backend/model pairs this policy needs. An engine is built once and shared."
        out = set()
        for j in JOBS:
            try: s = self.spec(j)
            except Exception: continue      # a job with nowhere to run needs no engine built
            out.add((s.backend, s.model_id))
        return out

    def summary(self):
        "The whole policy in one block, for `/model` with no argument, including anything that moved."
        rows = []
        for j in JOBS:
            try: row = model_note(self.spec(j))
            except Exception as e: row = f'unavailable ({agent_err(e)})'
            rows.append(f'{j:11} {row}' + (f'  [{self.notes[j]}]' if j in self.notes else ''))
        return '\n'.join(rows)

By default the expensive job is the user's model and the cheap ones are local.

In [ ]:
r = Routing(turn='sonnet')
r.name_for('turn'), r.name_for('summary'), r.name_for('completion')

('sonnet', 'sonnet', 'gemma-e4b')

In [ ]:
# a summary of this conversation follows the model holding it; the cheap jobs share `oneshot`
test_eq((r.name_for('subagent'), r.name_for('summary')), ('gpt-4.1', 'sonnet'))
test_eq((r.name_for('completion'), r.name_for('classify')), (DFLT_LOCAL, DFLT_LOCAL))

`set` validates before it stores. A misspelled model fails at the `/model` command instead of during the next turn.

In [ ]:
r.set('opus', 'summary')
test_fail(lambda: r.set('sonnnet'), contains='unknown model')
r.name_for('summary')

'opus'

`backends` collapses the policy to the distinct engines it needs, which is what lets one loaded model serve more than one job.

In [ ]:
Routing(turn='sonnet').backends()

{('claude', 'claude-sonnet-5'),
 ('litert', 'litert-community/gemma-4-E4B-it-litert-lm'),
 ('remote', 'openai/gpt-4.1')}

`summary` renders the whole policy, for `/model` with no argument.

In [ ]:
print(Routing(turn='sonnet').summary())

turn        sonnet · cloud · 200k ctx · in: text image
oneshot     gemma-e4b · local · 16k ctx · in: text image audio
inline      gemma-e4b · local · 16k ctx · in: text image audio
completion  gemma-e4b · local · 16k ctx · in: text image audio
classify    gemma-e4b · local · 16k ctx · in: text image audio
summary     sonnet · cloud · 200k ctx · in: text image
subagent    gpt-4.1 · cloud · 1047k ctx · in: text image · via tool: image


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()